---
title: "Walkthroughs and Exercises for Machine Learning for Data Analytics with Python"
author: "Dr. Chester Ismay"
format:
  html:
    theme: flatly
    toc: true
    toc-floating: true
  ipynb: default
engine: jupyter
execute:
  warning: false
---

<!-- https://ismay-oreilly-ml.netlify.app/exercises_solutions.html -->

In [ ]:
import pandas as pd

# Display all columns
pd.set_option('display.max_columns', None)

# Display all outputs from each cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Intro: Getting Started with Machine Learning for Data-Driven Decisions

## Walkthrough: Setting Up the Python Environment for ML

If you haven't already installed Python, Jupyter, and the necessary packages, there are instructions on the course repo in the README to do so [here](https://github.com/ismayc/oreilly-data-analysis-with-python/blob/main/README.md).

You can also install the packages directly in a Jupyter notebook with

In [ ]:
#| eval: false
# (no pip needed in this in-browser kernel) pip install pandas seaborn matplotlib scikit-learn mlxtend

If you aren't able to do this on your machine, you may want to check out [Google Colab](https://colab.research.google.com/). It's a free service that allows you to run Jupyter notebooks in the cloud. Alternatively, I've set up some temporary notebooks on Binder [here](https://mybinder.org/v2/gh/ismayc/oreilly-ml-for-data-analytics-with-python/HEAD?urlpath=%2Fdoc%2Ftree%2Fexercises.ipynb) that you can work with online as well.

Run the following code to check that each of the needed packages are installed. If you get an error, you may need to install the package(s) again.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.model_selection import GridSearchCV

In [ ]:
# Load dataset
telco_churn_raw = pd.read_csv('telco-customer-churn.csv')

## Exercise: Setting Up the Python Environment

By completing this exercise, you will be able to

- Import necessary Python packages
- Check for successful package loading
- Load datasets into Python

Follow the instructions above in Walkthrough to check for correct installation
of necessary packages.

In [ ]:
# Load dataset
marketing_campaign_raw = pd.read_csv('marketing_campaign.csv')

---

# Module 1: Data Understanding and Preprocessing for Machine Learning


## Walkthrough 1.1: Exploring and Preprocessing Data with Pandas & Seaborn

### Inspect a dataset using Pandas


In [ ]:
# Inspect data structure
telco_churn_raw
telco_churn_raw.info()

In [ ]:
# Check for missing values
telco_churn_raw.isnull().sum()

# Check for duplicate rows
telco_churn_raw.duplicated().sum()

### Handle missing values and clean data

In [ ]:
# Make a copy of the data to fix and clean
telco_churn = telco_churn_raw.copy()

# Handle missing values
telco_churn['MonthlyCharges'] = telco_churn['MonthlyCharges']\
  .fillna(telco_churn['MonthlyCharges'].median())
telco_churn['AvgServiceUsageScore'] = telco_churn['AvgServiceUsageScore']\
  .fillna(telco_churn['AvgServiceUsageScore'].median())
telco_churn.info()

In [ ]:
# Standardize column formats (e.g., convert Yes/No to binary for a few columns)
telco_churn['SeniorCitizen'] = telco_churn['SeniorCitizen'].astype('category')
telco_churn['Churn'] = telco_churn['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# Summarize statistics
telco_churn.describe(include='all')

### Create visualizations to identify key business trends

In [ ]:
# Visualize distributions and relationships
sns.violinplot(data=telco_churn, x='Churn', y='tenure', inner='quartile')
plt.title('Violin Plot of Tenure by Churn')
plt.show();

In [ ]:
sns.histplot(data=telco_churn, x='AvgServiceUsageScore')
plt.title('Histogram of Average Service Usage Score')
plt.show();

In [ ]:
sns.histplot(data=telco_churn, x='MonthlyCharges')
plt.title('Histogram of Monthly Charges')
plt.show();

In [ ]:
sns.scatterplot(data=telco_churn, x='AvgServiceUsageScore', y='MonthlyCharges')
plt.title('Average Service Usage Score vs. Monthly Charges')
plt.show();

In [ ]:
sns.countplot(data=telco_churn, x='InternetService', hue='Churn')
plt.title('Churn Rate by Internet Service Type')
plt.show();

---



## Exercise 1.1: Exploring and Preprocessing Data with Pandas & Seaborn

### Inspect a dataset using Pandas

In [ ]:
# Inspect data structure
marketing_campaign_raw
marketing_campaign_raw.info()

In [ ]:
# Check for missing values
marketing_campaign_raw.isnull().sum()

# Check for duplicate rows
marketing_campaign_raw.duplicated().sum()

### Handle missing values and clean data

> **Common Pitfall:** The `Income` column has missing values. If you skip this step and try to build models later, you'll get errors. Always re-check `isnull().sum()` after cleaning to verify it returns zeros.

In [ ]:
# Create a clean working copy
marketing_campaign = marketing_campaign_raw.copy()

# Handle missing values of Income
marketing_campaign['Income'] = marketing_campaign['Income']\
  .fillna(
    marketing_campaign['Income'].median()
  )

# Add new features
marketing_campaign["TotalChildren"] = marketing_campaign["Kidhome"] + marketing_campaign["Teenhome"]
marketing_campaign["TotalSpent"] = (
    marketing_campaign["MntWines"] + marketing_campaign["MntFruits"] +
    marketing_campaign["MntMeatProducts"] + marketing_campaign["MntFishProducts"] +
    marketing_campaign["MntSweetProducts"] + marketing_campaign["MntGoldProds"]
)

# Convert to datetime
marketing_campaign["Dt_Customer"] = pd.to_datetime(marketing_campaign["Dt_Customer"], errors='coerce')
marketing_campaign.info()

In [ ]:
# Summarize structure
marketing_campaign.describe(include='all')

### Create visualizations to identify key business trends

In [ ]:
sns.violinplot(data=marketing_campaign, x='Response', y='Income', inner='quartile')
plt.title('Violin Plot of Income by Campaign Response')
plt.show();

In [ ]:
sns.histplot(data=marketing_campaign, x='TotalSpent')
plt.title('Histogram of Total Spent')
plt.show();

In [ ]:
sns.scatterplot(data=marketing_campaign, x='Income', y='TotalSpent',
                hue='Response', alpha=0.7)
plt.title('Income vs. Total Spent by Campaign Response')
plt.show();

In [ ]:
sns.countplot(data=marketing_campaign, x='Education', hue='Response')
plt.title('Campaign Response by Education Level')
#plt.xticks(rotation=30)
plt.show();

### Interpretation Questions

1. Looking at the violin plot of Income by Response, which group (responders or non-responders) shows more variability in income?
2. Based on your scatter plot, do higher-income customers tend to spend more? Is this relationship strong or weak?
3. Which education level shows the highest response rate? What marketing implications might this have?

### Self-Check

By the end of this module, you should be able to:

- [ ] Load a dataset and inspect its structure with `.info()` and `.describe()`
- [ ] Identify and handle missing values before modeling
- [ ] Create at least 3 types of visualizations (histogram, scatter, violin/box)
- [ ] Articulate one business insight from your exploratory analysis

---


# Module 2: Supervised Learning for Business Decisions

## Walkthrough 2.1: Build a Regression Model for Pricing Optimization

### Split the data into training and validation sets

> **Common Pitfall:** Fitting `StandardScaler` on the full dataset before the train/test split leaks information from the validation set into training. In a strict workflow, fit the scaler on the training data only (or wrap it in a `Pipeline`) so the validation set stays truly unseen.

In [ ]:
X = telco_churn[['AvgServiceUsageScore']]  # predictor
y = telco_churn['MonthlyCharges']          # target

# Reset index to avoid potential issues later
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

# Best practice when working with linear models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
  X_scaled, y, test_size=0.2, random_state=2025
)

### Train a linear regression model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"Intercept: {lr.intercept_:.2f}")
print(f"Coefficient (usage_scaled -> price): {lr.coef_[0]:.2f}")

> **Common Pitfall:** Because the predictor was standardized before fitting, this coefficient is the change in price per one-standard-deviation change in usage, not per raw unit. Keep that in mind when explaining the number to stakeholders, and convert back to raw units if you need a real-world-unit effect.

### Evaluate model performance on the validation set

In [ ]:
y_pred = lr.predict(X_val)

r2 = r2_score(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)

print(f"R-squared: {r2:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")

---

## Exercise 2.1: Build a Regression Model for Pricing Optimization

### Split the data into training and validation sets

In [ ]:
# Predictor and target
X_m = marketing_campaign[['Income']]  # predictor
y_m = marketing_campaign['TotalSpent'] # target

# Reset index to avoid potential issues later
X_m = X_m.reset_index(drop=True)
y_m = y_m.reset_index(drop=True)

# Standardize the predictor
scaler_m = StandardScaler()
X_m_scaled = scaler_m.fit_transform(X_m)

# Train-test split
X_m_train, X_m_val, y_m_train, y_m_val = train_test_split(
    X_m_scaled, y_m, test_size=0.2, random_state=2025
)

### Train a linear regression model

In [ ]:
lr_m = LinearRegression()
lr_m.fit(X_m_train, y_m_train)

print(f"Intercept: {lr_m.intercept_:.2f}")
print(f"Coefficient (Income_scaled -> TotalSpent): {lr_m.coef_[0]:.2f}")

### Evaluate model performance on the validation set

In [ ]:
y_m_pred = lr_m.predict(X_m_val)

r2_m = r2_score(y_m_val, y_m_pred)
mae_m = mean_absolute_error(y_m_val, y_m_pred)

print(f"R-squared: {r2_m:.2f}")
print(f"Mean Absolute Error: {mae_m:.2f}")

### Interpretation Questions

1. Is your R-squared higher or lower than the telco churn model? What might explain the difference?
2. If MAE is $200, what does that mean in practical terms for predicting customer spending?
3. Would you trust this model for making budget decisions? Why or why not?

---

## Walkthrough 2.2: Implement a Classification Model for Customer Churn

### Split the data into training and validation sets

> **Common Pitfall:** Churn and campaign response are imbalanced (far more 0s than 1s), so accuracy alone can be misleading -- a model that always predicts the majority class can still look "accurate." Report precision and recall alongside accuracy to see how well the model catches the minority class.

In [ ]:
# Select relevant features
features = ['tenure', 'SeniorCitizen', 'ServiceCount', 'InternetScore', 'AvgServiceUsageScore']
X = telco_churn[features]
y = telco_churn['Churn']

# Scaling is not as important for tree-based models since they are not sensitive to
# one feature having a larger scale than another

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2025)

### Train a Random Forest classification model

In [ ]:
# Train Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=2025)
clf.fit(X_train, y_train)

### Evaluate model performance on the validation set

In [ ]:
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(f"Precision: {precision_score(y_test, y_pred):.2f}")
print(f"Recall: {recall_score(y_test, y_pred):.2f}")

cm = confusion_matrix(y_test, y_pred)

labels = ['No Churn', 'Churn']
cm_telco_churn = pd.DataFrame(cm, index=labels, columns=labels)

print("\nConfusion Matrix (formatted):")
print(cm_telco_churn)

### Quick Reference: When to Use Which Metric

| Situation | Metric | Why |
|-----------|--------|-----|
| Predicting continuous values | R-squared, MAE | Measures prediction error |
| Balanced classes | Accuracy | Overall correctness |
| Cost of false positives is high | Precision | Minimize wrong positive predictions |
| Cost of false negatives is high | Recall | Catch all actual positives |
| Need balance | F1-Score | Harmonic mean of precision/recall |

---

## Exercise 2.2: Implement a Classification Model for Customer Churn

### Split the data into training and validation sets

In [ ]:
# Select relevant features
features_m = ['Income', 'TotalSpent', 'TotalChildren']
X_m_class = marketing_campaign[features_m]
y_m_class = marketing_campaign['Response']

# Split into training and validation sets
X_m_train_class, X_m_val_class, y_m_train_class, y_m_val_class = train_test_split(
    X_m_class, y_m_class, test_size=0.2, random_state=2025
)

### Train a Random Forest classification model

In [ ]:
# Train Random Forest classifier
clf_m = RandomForestClassifier(n_estimators=100, random_state=2025)
clf_m.fit(X_m_train_class, y_m_train_class)

### Evaluate model performance on the validation set

In [ ]:
y_m_pred_class = clf_m.predict(X_m_val_class)

print(f"Accuracy: {accuracy_score(y_m_val_class, y_m_pred_class):.2f}")
print(f"Precision: {precision_score(y_m_val_class, y_m_pred_class):.2f}")
print(f"Recall: {recall_score(y_m_val_class, y_m_pred_class):.2f}")

labels = ['No Response', 'Response']
cm_marketing_response = pd.DataFrame(confusion_matrix(y_m_val_class, y_m_pred_class),
                                     index=labels, columns=labels)

print("\nConfusion Matrix (formatted):")
print(cm_marketing_response)

### Interpretation Questions

1. Compare your precision and recall. Which is higher, and what does that imply about the model's tendencies?
2. Looking at your confusion matrix, is the model better at identifying responders or non-responders?
3. If running a marketing campaign costs $50 per contact, which metric matters more: precision or recall?

### Self-Check

By the end of this module, you should be able to:

- [ ] Explain the difference between regression and classification
- [ ] Split data into training and validation sets with a fixed `random_state`
- [ ] Interpret R-squared, MAE, accuracy, precision, and recall
- [ ] Explain why we don't evaluate a model on its training data
- [ ] Recognize when accuracy is misleading on imbalanced classes

---

# Module 3: Unsupervised Learning and Pattern Discovery in Business

## Walkthrough 3.1: Exploring K-Means Clustering for Customer Segmentation

### Apply K-Means clustering to segment customers

> **Common Pitfall:** K-means uses Euclidean distance, so an unscaled feature with a large range (like `MonthlyCharges`) will dominate the clusters. Always standardize features before clustering, and remember the cluster labels are arbitrary integers with no inherent order.

In [ ]:
# Select relevant features
telco_churn['ContractType'] = telco_churn['Contract'].map({
    'Month-to-month': 0, 'One year': 1, 'Two year': 2
})
features = ['tenure', 'ServiceCount', 'AvgServiceUsageScore', 'MonthlyCharges', 'InternetScore', 'ContractType']
X = telco_churn[features]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Determine the optimal number of clusters using the Elbow Method

In [ ]:
inertia = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=2025)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show();

### Verify using the silhouette score (optional but recommended)

In [ ]:
# Evaluate silhouette scores for k=3 to k=6
silhouette_scores = {}

for k in range(3, 7):
    kmeans = KMeans(n_clusters=k, random_state=2025)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores[k] = score

silhouette_scores

### Fit K-means and assign cluster labels to each customer

In [ ]:
# Let's assume the elbow suggested k=3
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=2025)
telco_churn['Cluster'] = kmeans.fit_predict(X_scaled)

### Visualize customer segments using a 2D plot

In [ ]:
# Visualize clusters in 2D space (using tenure and MonthlyCharges)
sns.scatterplot(data=telco_churn,  x='tenure', y='AvgServiceUsageScore',
                hue='Cluster', palette='viridis')
plt.title('Customer Segments via K-Means')
plt.show();

---

## Exercise 3.1: Exploring K-Means Clustering for Customer Segmentation

### Apply K-Means clustering to segment customers

In [ ]:
# Select relevant features
features_cluster = ['Income', 'TotalSpent', 'TotalChildren']
X_cluster = marketing_campaign[features_cluster]

# Standardize features
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

### Determine the optimal number of clusters using the Elbow Method

In [ ]:
inertia = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=2025)
    kmeans.fit(X_cluster_scaled)
    inertia.append(kmeans.inertia_)

plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show();

### Verify using the silhouette score (optional but recommended)

In [ ]:
silhouette_scores = {}

for k in range(3, 7):
    kmeans = KMeans(n_clusters=k, random_state=2025)
    labels = kmeans.fit_predict(X_cluster_scaled)
    score = silhouette_score(X_cluster_scaled, labels)
    silhouette_scores[k] = score

silhouette_scores

### Fit K-means and assign cluster labels to each customer

In [ ]:
# Assume optimal k is 5 (can be adjusted based on elbow or silhouette)
optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, random_state=2025)
marketing_campaign['Cluster'] = kmeans_final.fit_predict(X_cluster_scaled)

### Visualize customer segments using a 2D plot

In [ ]:
import matplotlib.ticker as ticker
sns.scatterplot(data=marketing_campaign, x='TotalChildren', y='TotalSpent',
                hue='Cluster', palette='viridis')
plt.title('Customer Segments via K-Means Clustering')

# Set x-axis to integer ticks only
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.show();

### Interpretation Questions

1. How did you decide on the optimal k? Did the elbow method and silhouette scores agree?
2. Looking at your 2D visualization, do the clusters seem well-separated or do they overlap?
3. Can you describe what "type" of customer each cluster might represent based on their TotalChildren and TotalSpent values?

---

## Walkthrough 3.2: Market Basket Analysis with Apriori Algorithm

### Prepare transactional data (services as items)

> **Common Pitfall:** Setting `min_support` too low floods you with rules (many spurious), while setting it too high can return no itemsets at all. Also, a rule can have high confidence simply because the consequent is popular -- always check `lift` (> 1) to confirm a real association rather than a coincidence.

In [ ]:
# Selecting binary service columns to act like "products"
service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PhoneService'
]

# Convert service columns to boolean (preference of apriori() function)
telco_churn_basket = telco_churn[service_cols].astype(bool)
telco_churn_basket

### Apply the Apriori algorithm to identify frequent itemsets

In [ ]:
frequent_itemsets = apriori(telco_churn_basket, min_support=0.2, use_colnames=True)
frequent_itemsets

### Generate association rules from frequent itemsets

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules

---

## Exercise 3.2: Market Basket Analysis with Apriori Algorithm

### Prepare transactional data (product categories as items)

In [ ]:
# Select binary columns representing product purchases
product_cols = [
    'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds'
]

# Convert product columns to binary: 1 if any amount was spent, else 0
basket = marketing_campaign[product_cols].map(lambda x: 1 if x > 0 else 0)

# Rename columns for cleaner output
basket.columns = [
    'Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold'
]

basket = basket.astype(bool)
basket.head()

---

### Apply the Apriori algorithm to identify frequent itemsets

In [ ]:
frequent_itemsets = apriori(basket, min_support=0.2, use_colnames=True)
frequent_itemsets.sort_values(by='support', ascending=False)

---

### Generate association rules from frequent itemsets

In [ ]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules = rules.sort_values(by='lift', ascending=False)
rules.reset_index(drop=True, inplace=True)
rules

### Interpretation Questions

1. Which product category appears in the most frequent itemsets? What does this suggest about purchasing patterns?
2. Find a rule with high confidence but low lift. Why might this rule be less useful despite high confidence?
3. Identify one actionable insight: what product bundle would you recommend based on these rules?

### Self-Check

By the end of this module, you should be able to:

- [ ] Explain the difference between supervised and unsupervised learning
- [ ] Scale features before clustering and explain why it matters
- [ ] Use the elbow method and silhouette score to choose k
- [ ] Interpret support, confidence, and lift in association rules
- [ ] Describe a business application for clustering and market basket analysis

---

# Module 4: Implementing and Evaluating ML Models

## Walkthrough 4.1: Exploring Cross-Validation for Model Evaluation

### Split data into training and validation sets

> **Common Pitfall:** Reporting a single train-test split can be lucky or unlucky depending on which rows land where. Cross-validation averages over several folds for a more honest estimate -- and forgetting `random_state` makes those folds (and your results) non-reproducible.

In [ ]:
# Features and target
X = telco_churn[['AvgServiceUsageScore', 'tenure', 'MonthlyCharges']]
y = telco_churn['Churn']

# Scale the features since logistic regression is a linear model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
  X_scaled, y, test_size=0.2, random_state=2025
)

### Train a classification model using logistic regression

In [ ]:
# Initialize logistic regression model
logreg = LogisticRegression(max_iter=1000, solver='liblinear')

### Apply k-fold cross-validation to evaluate model performance

In [ ]:
# Cross-validate with multiple metrics
cv_results = cross_validate(
    logreg,
    X_train,
    y_train,
    cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

### Compare metrics across folds

In [ ]:
def show_metric(name, results):
    scores = results[f"test_{name}"]
    print(f"{name.capitalize()}: {scores.mean():.3f} ± {scores.std():.3f}")

# Average performance across folds
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    show_metric(metric, cv_results)

**Interpretation**

Accuracy -> Overall correctness of predictions

Precision -> How many predicted churns were actual churns

Recall -> How many churns were correctly identified

F1-Score -> Balances precision & recall

Cross-validation ensures that your model generalizes better to unseen data by reducing the risk of overfitting on a single split.

---

## Exercise 4.1: Exploring Cross-Validation for Model Evaluation

### Split data into training and validation sets

In [ ]:
# Features and target
X_m = marketing_campaign[['Income', 'TotalSpent', 'TotalChildren']]
y_m = marketing_campaign['Response']

# Scale features
scaler = StandardScaler()
X_m_scaled = scaler.fit_transform(X_m)

# Train-test split
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_m_scaled, y_m, test_size=0.2, random_state=2025
)

### Train a classification model using logistic regression

In [ ]:
# Initialize logistic regression model
logreg_m = LogisticRegression(max_iter=1000, solver='liblinear')

### Apply k-fold cross-validation to evaluate model performance

In [ ]:
# 5-fold cross-validation with multiple metrics
cv_results_m = cross_validate(
    logreg_m,
    X_m_train,
    y_m_train,
    cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

### Compare metrics across folds

In [ ]:
# Average performance across folds
for m in ["accuracy", "precision", "recall", "f1"]:
    show_metric(m, cv_results_m)

**Interpretation**

Accuracy -> Overall correctness of predictions

Precision -> How many predicted responders were actual responders

Recall -> How many actual responders were correctly identified

F1-Score -> Harmonic mean of precision and recall

### Interpretation Questions

1. How consistent are your metrics across folds? (Look at the standard deviation values.)
2. Which metric shows the most variability? What might cause this?
3. Compare your cross-validation results to the single train-test split in Exercise 2.2. Are they similar?

---

## Walkthrough 4.2: Hyperparameter Tuning with Grid Search

### Train a Random Forest classifier

> **Common Pitfall:** Tuning hyperparameters and then evaluating on the same data leaks the answer and overstates performance. The honest grid-search workflow selects parameters using cross-validation on the training set, then reports the final score on a held-out test set the search never saw.

In [ ]:
# Features and target
X = telco_churn[['AvgServiceUsageScore', 'tenure', 'MonthlyCharges']]
y = telco_churn['Churn']

# Initialize Random Forest
rf = RandomForestClassifier(random_state=2025)

### Apply grid search to find optimal hyperparameters

In [ ]:
# Define the hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 8],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# GridSearchCV setup
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

# Run grid search
grid_search.fit(X, y)

### Evaluate model improvement using accuracy and recall

In [ ]:
# Predict using the best model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X)

# Evaluate
print(f"Best Accuracy: {accuracy_score(y, y_pred):.3f}")
print(f"Best Recall: {recall_score(y, y_pred):.3f}")

### Interpret the best hyperparameter combination

In [ ]:
print("Best Parameters Found:", grid_search.best_params_)

---

## Exercise 4.2: Hyperparameter Tuning with Grid Search

### Train a Random Forest classifier

In [ ]:
# Features and target
X_m = marketing_campaign[['Income', 'TotalSpent', 'TotalChildren']]
y_m = marketing_campaign['Response']

# Initialize Random Forest
rf_m = RandomForestClassifier(random_state=2025)

### Apply grid search to find optimal hyperparameters

In [ ]:
# Define the hyperparameter grid
param_grid_m = {
    'n_estimators': [100, 200],
    'max_depth': [4, 8],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# GridSearchCV setup
grid_search_m = GridSearchCV(
    estimator=rf_m,
    param_grid=param_grid_m,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

# Run grid search
grid_search_m.fit(X_m, y_m)

### Evaluate model improvement using accuracy and recall

In [ ]:
# Predict using the best model
best_rf_m = grid_search_m.best_estimator_
y_m_pred = best_rf_m.predict(X_m)

# Evaluate
print(f"Best Accuracy: {accuracy_score(y_m, y_m_pred):.3f}")
print(f"Best Recall: {recall_score(y_m, y_m_pred):.3f}")

### Interpret the best hyperparameter combination

In [ ]:
print("Best Parameters Found:", grid_search_m.best_params_)

### Interpretation Questions

1. Did hyperparameter tuning improve recall compared to the default model in Exercise 2.2?
2. Look at the best parameters found. Are they at the edges of your grid (suggesting you should expand the search)?
3. Was the computational cost of grid search worth the performance improvement?

### Self-Check

By the end of this module, you should be able to:

- [ ] Explain why cross-validation is better than a single train-test split
- [ ] Interpret the mean and standard deviation of cross-validation scores
- [ ] Define what hyperparameters are and why we tune them
- [ ] Use GridSearchCV to find optimal model settings
- [ ] Explain why tuning and reporting on the same data inflates performance

---

# Bonus Challenge: End-to-End ML Pipeline

If you finish early or want additional practice, try this integration challenge using the `marketing_campaign` data:

**Goal:** Build the best model to predict `Response` using everything you've learned.

1. **Feature Engineering:** Create at least one new feature beyond TotalChildren and TotalSpent (e.g., spending per child, years as customer from Dt_Customer)

2. **Model Comparison:** Train both Logistic Regression and Random Forest, use cross-validation to compare them fairly

3. **Optimization:** Use GridSearchCV on your better-performing model

4. **Interpretation:** Write 2-3 sentences explaining which model you'd recommend and why

This is open-ended. There's no single right answer. The goal is to practice the full workflow.

In [ ]:
# Your code here